In [18]:
# I think the paths from racke are sorted -- ok maybe I sorted them
homedir = "/mnt/mirabelle/az6922_homedir/"
fromnpfile = f"{homedir}DRing/src/emp/datacentre/rawrackepathfiles/dring80/racke1"
tonpfile = f"{homedir}DRing/src/emp/datacentre/evalnetpathfiles/netpath_dring_80_64_racke1four.np"

In [23]:
import re

netpaths = list()
for i in range(80):
    netpaths.append(list())
    for j in range(80):
        netpaths[i].append(dict())
with open(fromnpfile, "r") as fr:
    lines = fr.readlines()
    fromsw = -1
    tosw = -1
    mynetpath = dict()
    for line in lines:
        if '->' in line:
            if fromsw!=-1 and tosw!=-1:
                netpaths[fromsw][tosw] = mynetpath
            tokens = line.split(':')[0].split('->')
            fromsw = int(re.search(r'\d+', tokens[0].strip()).group())
            tosw = int(re.search(r'\d+', tokens[1].strip()).group())
            mynetpath = dict()
        elif '@' in line:
            tokens = line.split('@')
            hoplist = numbers =list(map(int, re.findall(r'\d+', tokens[0].strip())))
            weight = float(tokens[1].strip())
            mynetpath[weight] = list(dict.fromkeys(hoplist)) # find unique hops
            # print(f"from {fromsw} to {tosw} weight {weight} path {list(dict.fromkeys(hoplist))}")
    netpaths[fromsw][tosw] = mynetpath

pathlimit = 4
with open(tonpfile, "w") as fw:
    for i in range(80):
        for j in range(80):
            if i==j:
                fw.write(f"{i} {j} 0\n")
            else:
                mynetpath = netpaths[i][j]
                sorted_mynetpath = dict(sorted(mynetpath.items(),reverse=True))
                fw.write(f"{i} {j} {pathlimit}\n")
                ip = 0
                for _,path in sorted_mynetpath.items():
                    if ip >= pathlimit:
                        break
                    else:
                        for ih, hop in enumerate(path[:-1]):
                            fw.write(f" {hop}->{path[ih+1]}")
                        ip += 1
                    fw.write("\n")